# Qdrant Server Mode (Docker)

This notebook demonstrates FFAI's Qdrant backend connected to a **local
Qdrant server** running in Docker.

- Auto-starts the container via `docker compose` if not already running
- Creates a collection, indexes documents, runs searches
- **Clears the collection and stops the container** at the end

Requires Docker. Uses synthetic embeddings (no API keys needed).

<div class="page-break"></div>

---

## Section 1: Start Qdrant Server

In [ ]:
import socket
import subprocess
import sys
import time
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

COMPOSE_FILE = str(_project_root / 'docker-compose.dev.yaml')
QDRANT_HOST = 'localhost'
QDRANT_PORT = 6333


def qdrant_is_running():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2)
    try:
        sock.connect((QDRANT_HOST, QDRANT_PORT))
        sock.close()
    except (ConnectionRefusedError, OSError):
        sock.close()
        return False
    try:
        from qdrant_client import QdrantClient

        c = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=5)
        c.get_collections()
        c.close()
        return True
    except Exception:
        return False


if not qdrant_is_running():
    print('Starting Qdrant container...')
    subprocess.run(
        ['docker', 'compose', '-f', COMPOSE_FILE, 'up', '-d', 'qdrant'],
        check=True,
    )
    for _i in range(30):
        if qdrant_is_running():
            break
        time.sleep(1)
    else:
        raise RuntimeError('Qdrant container did not start within 30s')
    print(f'Container started (waited {_i+1}s)')
    _STARTED_CONTAINER = True
else:
    print('Qdrant server already running')
    _STARTED_CONTAINER = False

<div class="page-break"></div>

---

## Section 2: Create the Store

Connect to the server on `localhost:6333`.

In [ ]:
import numpy as np

from ffai.rag.stores import get_store

DIM = 128

store = get_store(
    backend='qdrant',
    collection_name='nb_server_demo',
    embedding_dim=DIM,
    host=QDRANT_HOST,
    port=QDRANT_PORT,
)

print(f'Backend: {store.name}')
print(f'Connected to: {QDRANT_HOST}:{QDRANT_PORT}')
print(f'Initial count: {store.count()}')

<div class="page-break"></div>

---

## Section 3: Index Documents

Load sample documents, chunk them, and index with synthetic embeddings.

In [ ]:
import asyncio

from examples._rag_data.download import download
from ffai.rag.splitters import get_chunker


def make_embedding(text: str, dim: int = DIM) -> list[float]:
    rng = np.random.default_rng(hash(text) & 0xFFFFFFFF)
    vec = rng.standard_normal(dim)
    return (vec / np.linalg.norm(vec)).tolist()


sample_docs = download()
total = 0
for name, path in sample_docs.items():
    text = path.read_text(encoding='utf-8')
    chunks = get_chunker('recursive', chunk_size=500, chunk_overlap=50).chunk(text)
    ids = [f'{name}_chunk{i}' for i in range(len(chunks))]
    texts = [c.content for c in chunks]
    embs = [make_embedding(t) for t in texts]
    metas = [{'source': name, 'chunking_strategy': 'recursive'} for _ in chunks]
    added = asyncio.run(store.aadd(ids, texts, embs, metas))
    total += added
    print(f'  {name}: {added} chunks')

print(f'\nTotal indexed: {total} chunks')
print(f'Sources: {store.list_sources()}')

<div class="page-break"></div>

---

## Section 4: Search with and without Filters

Run queries against the server-side Qdrant instance.

In [ ]:
queries = [
    ('asynchronous programming', None),
    ('bearer token authentication', {'source': 'api_docs.md'}),
    ('error handling', {'source': 'python_tutorial.md'}),
]

for qtext, where in queries:
    emb = make_embedding(qtext)
    hits = asyncio.run(store.asearch(emb, top_k=3, where=where))
    filter_str = f' (filter: {where})' if where else ''
    print(f'Query: "{qtext}"{filter_str}')
    for i, h in enumerate(hits, 1):
        print(f'  {i}. [{h.score:.4f}] {h.content[:80]}...')
        print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 5: Cleanup

Clear the collection and stop the Docker container (if we started it).

In [ ]:
store.clear()
print(f'After clear: {store.count()} chunks')

if _STARTED_CONTAINER:
    print('\nStopping Qdrant container...')
    subprocess.run(
        ['docker', 'compose', '-f', COMPOSE_FILE, 'down'],
        check=True,
    )
    print('Container stopped.')
else:
    print('\nContainer was already running — leaving it up.')

print('\nCleanup complete.')